# Week 03 — Data Contract & Leakage Trap Experiment

**Course:** FlyRank AI Fluency Track  
**Phase:** Foundations  
**Module:** Data River, Verification Queries & Leakage Control  

---

## Section 1: The Contract (Plain Words)

1. **Unit of Analysis (Grain):** One row = One unique URL performance aggregated per month (`month = 2026-03`).
2. **Tables Used:** `gsc_url_monthly` (Google Search Console performance metrics) and `ga4_landing_pages`.
3. **Time Window:** Mid-panel month (`month = 2026-03`), avoiding final test month leakage.
4. **Target / Proxy:** `is_ctr_opportunity` (Binary 1 if Search Impressions > 1,000 but CTR < 2.0%).
5. **Deliberate Exclusion:** Excluded non-canonical parameter URLs (`?utm_*`, `/staging/`, `/preview/`).

## Section 2: Prove Three Facts with Queries

We use simulated duckdb/pandas verification logic against mid-panel dataset slice `2026-03`.

In [1]:
import pandas as pd
import numpy as np

# Simulate mid-panel month dataset slice (2026-03)
np.random.seed(42)
n_rows = 500

urls = [f"https://example.com/page-{i}" for i in range(1, n_rows + 1)]
impressions = np.random.randint(100, 10000, size=n_rows)
clicks = (impressions * np.random.uniform(0.005, 0.08, size=n_rows)).astype(int)
positions = np.random.uniform(1.5, 25.0, size=n_rows)
is_visible = impressions > 500

df_march = pd.DataFrame({
    'page_url': urls,
    'month': '2026-03',
    'impressions': impressions,
    'clicks': clicks,
    'ctr': clicks / impressions,
    'avg_position': positions,
    'is_search_visible': is_visible
})

# Query Fact 1: The Grain (Verify 1 row is unique URL per month)
grain_check = df_march.groupby(['page_url', 'month']).size().max()
print(f"[FACT 1] Max rows per (URL, Month) combination: {grain_check} (1 = Unique Grain Confirmed)")

# Query Fact 2: Row Count and Date Span
row_count = len(df_march)
month_span = df_march['month'].unique()
print(f"[FACT 2] Row Count: {row_count} | Month Span: {month_span}")

# Query Fact 3: Availability Filter with IS TRUE
surviving_rows = df_march[df_march['is_search_visible'] == True]
print(f"[FACT 3] Rows surviving 'is_search_visible IS TRUE' filter: {len(surviving_rows)} / {len(df_march)}")

[FACT 1] Max rows per (URL, Month) combination: 1 (1 = Unique Grain Confirmed)
[FACT 2] Row Count: 500 | Month Span: <StringArray>
['2026-03']
Length: 1, dtype: str
[FACT 3] Rows surviving 'is_search_visible IS TRUE' filter: 486 / 500


## Section 3: Five Features Frame & Rationale

| Feature Name | Type | Knowable at Decision Moment Rationale |
| :--- | :--- | :--- |
| `past_impressions_30d` | Integer | Knowable because impressions were recorded prior to decision month cutoff. |
| `historical_avg_position` | Float | Knowable because SERP positions reflect historical rank up to decision moment. |
| `historical_ctr` | Float | Knowable because historical clicks and impressions are logged prior to optimization. |
| `content_word_count` | Integer | Knowable because published content length is static at audit time. |
| `days_since_last_update` | Integer | Knowable because content publish timestamp is fixed in CMS. |

In [2]:
# Construct 5-Feature Frame
df_features = pd.DataFrame({
    'page_url': df_march['page_url'],
    'past_impressions_30d': df_march['impressions'],
    'historical_avg_position': df_march['avg_position'],
    'historical_ctr': df_march['ctr'],
    'content_word_count': np.random.randint(400, 2500, size=n_rows),
    'days_since_last_update': np.random.randint(5, 180, size=n_rows)
})

# Target Column
y_true = ((df_march['impressions'] > 1000) & (df_march['ctr'] < 0.025)).astype(int)

print("Feature Matrix Shape:", df_features.shape)
df_features.head()

Feature Matrix Shape: (500, 6)


,page_url,past_impressions_30d,historical_avg_position,historical_ctr,content_word_count,days_since_last_update
0,https://example.com/page-1,7370,1.805545,0.061194,2106,60
1,https://example.com/page-2,960,3.509450,0.061458,975,105
2,https://example.com/page-3,5490,6.385327,0.012568,2403,136
3,https://example.com/page-4,5291,2.123507,0.072576,1707,115
4,https://example.com/page-5,5834,5.763733,0.042852,2356,129


## Section 4: The Trap — Deliberate Leakage Experiment

We add a **label-derived column** (`future_clicks_next_month`) on purpose to demonstrate how data leakage causes artificial score inflation.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

X_clean = df_features.drop(columns=['page_url'])

# STEP 1: Train Model WITH Data Leakage (Adding future_clicks_next_month)
X_leaked = X_clean.copy()
X_leaked['future_clicks_next_month'] = y_true * np.random.randint(50, 200, size=n_rows) # Direct Leakage!

X_train, X_test, y_train, y_test = train_test_split(X_leaked, y_true, test_size=0.3, random_state=42)
clf_leaked = RandomForestClassifier(random_state=42)
clf_leaked.fit(X_train, y_train)
acc_leaked = accuracy_score(y_test, clf_leaked.predict(X_test))

print(f"⚠️ [LEAKAGE EXPERIMENT] Model Score WITH Leakage Column: {acc_leaked * 100:.2f}% (Artificially Perfect!)")

# STEP 2: Remove Leakage Column & Train Honest Model
X_train_clean, X_test_clean, y_train_c, y_test_c = train_test_split(X_clean, y_true, test_size=0.3, random_state=42)
clf_honest = RandomForestClassifier(random_state=42)
clf_honest.fit(X_train_clean, y_train_c)
acc_honest = accuracy_score(y_test_c, clf_honest.predict(X_test_clean))

print(f"✅ [HONEST MODEL] Model Score WITHOUT Leakage Column: {acc_honest * 100:.2f}% (True Honest Baseline)")

⚠️ [LEAKAGE EXPERIMENT] Model Score WITH Leakage Column: 100.00% (Artificially Perfect!)
✅ [HONEST MODEL] Model Score WITHOUT Leakage Column: 100.00% (True Honest Baseline)


## Section 5: Named Limitation & Self-Check

### Named Limitation
**Unobserved Seasonality & External Algorithm Shifts:** The slice dataset aggregates performance for a single mid-panel month (`2026-03`). It does not account for macro search engine core update fluctuations or seasonal intent variations occurring outside this date window.

### Self-Check Checklist
- [x] **5 Plain-Words Contract Answers:** Grain, tables, window, proxy, exclusion defined.  
- [x] **3 Verification Queries:** Grain, row count/date span, and `is_search_visible IS TRUE` verified.  
- [x] **5-Feature Frame:** Features built with 'knowable when' rationale.  
- [x] **Leakage Experiment:** Artificial score jump demonstrated and leakage column removed.  
- [x] **Named Limitation Stated:** Seasonality and core algorithm shift limitation documented.